In [102]:
%reset -f

In [103]:
from get_data import *
from online_forecasting import *
from dash_plotter import DashRealTimePlotter
import warnings
import logging

import pandas as pd
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
import pickle
import os


import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report

warnings_logger = logging.getLogger('warnings')
warnings_logger.setLevel(logging.WARNING)
warning_handler = logging.FileHandler('warnings.log')
warning_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
warnings_logger.addHandler(warning_handler)

def warning_handler_func(message, category, filename, lineno, file=None, line=None):
    warnings_logger.warning(f"{category.__name__}: {message} (File: {filename}, Line: {lineno})")

warnings.showwarning = warning_handler_func

In [108]:
def encode_port_column_names_improved(columns, begin_with='Interface Gi'):
    port_numbers = []
    encoded_columns = []

    for col in columns:
        if col.startswith(begin_with):
            try:
                # More robust extraction
                first_part = col.split('Gi')[1]
                last_part = first_part.split('(')[0]
                port_num_str = last_part.replace("/", "")
                port_num = int(port_num_str)
                if port_num not in port_numbers:
                    port_numbers.append(port_num)
                if ": operational status" in col.lower():
                    encoded_columns.append(f"interface_status_{port_num}")
                else:
                    if "received" in col.lower():
                        encoded_columns.append(f"bits_received_{port_num}")
                    elif "sent" in col.lower():
                        encoded_columns.append(f"bits_sent_{port_num}")
                    else:
                        encoded_columns.append(col)  
            except (ValueError, IndexError) as e:
                print(f"Warning: Could not parse port number from column '{col}': {e}")
                encoded_columns.append(col)  # Keep original name
        else:
            encoded_columns.append(col)  
    
    return encoded_columns, sorted(port_numbers)

# encode statuses with binaries

def encode_binary_improved(df, down_value=2, temperature_threshold=50, temp_bit=8):
    
    # Pre-compute columns outside the apply function
    status_columns = [col for col in df.columns if 'status' in col.lower()]
    temp_candidates = [col for col in df.columns if 'temperature' in col.lower()]
    temperature_column = temp_candidates[0] if temp_candidates else None
    
    # Pre-extract port numbers for better performance
    port_mapping = {}
    for col in status_columns:
        if 'interface_status_' in status_columns:
            port_num = int(col.replace('interface_status_', ''))
            port_mapping[col] = port_num

    def create_binary_encoding(row):
        binary_value = 0
        
        # More efficient port status encoding
        for col, port_num in port_mapping.items():
            if row[col] == down_value:
                binary_value |= (1 << (port_num - 1))
        
        # Temperature check
        if temperature_column and not pd.isna(row[temperature_column]):
            if row[temperature_column] > temperature_threshold:
                binary_value |= (1 << temp_bit)
        
        return binary_value
    
    df_labeled = df.copy()
    df_labeled['binary_status'] = df.apply(create_binary_encoding, axis=1)
    
    return df_labeled, temp_bit

# label decoder

def decode_binary_improved(label, port_numbers, temp_bit=8):
    if not isinstance(label, (int, np.integer)):
        raise ValueError(f"Label must be an integer, got {type(label)}")
    
    if label < 0:
        raise ValueError(f"Label must be non-negative, got {label}")
    
    # Check for all OK condition
    if label == 0:
        return "All systems OK"
    
    # Decode port statuses
    down_ports = []
    for port in sorted(port_numbers):
        if label & (1 << (port - 1)):
            down_ports.append(port)
    
    # Decode temperature
    temperature_nok = bool(label & (1 << temp_bit))
    
    # Build readable status message
    status_parts = []
    
    if down_ports:
        if len(down_ports) == 1:
            status_parts.append(f"Port {down_ports[0]} down")
        else:
            ports_str = ', '.join(map(str, down_ports))
            status_parts.append(f"Ports {ports_str} down")
    
    if temperature_nok:
        status_parts.append("Temperature NOK")
    
    return "; ".join(status_parts) if status_parts else "Unknown status"

# balance small classes in df_labeled by downsampling majority classes

def merge_small_classes(df, label_col='label', threshold=10, other_label='other'):
    class_counts = df[label_col].value_counts()
    small_classes = class_counts[class_counts < threshold].index
    df_merged = df.copy()
    df_merged[label_col] = df_merged[label_col].apply(lambda x: other_label if x in small_classes else x)
    return df_merged

# balance the classes in labeled_df by downsampling majority classes
def balance_classes(df, label_col='label', random_state=42):
    class_counts = df[label_col].value_counts()
    min_count = class_counts.min()

    balanced_frames = []
    for cls in class_counts.index:
        cls_df = df[df[label_col] == cls]
        balanced_cls_df = resample(cls_df, 
                                   replace=False, 
                                   n_samples=min_count, 
                                   random_state=random_state)
        balanced_frames.append(balanced_cls_df)
    
    # Concatenate and shuffle
    balanced_df = pd.concat(balanced_frames).sample(frac=1, random_state=random_state).reset_index(drop=True)
    return balanced_df

def create_windows(X, y, window_size):
    Xs, ys = [], []
    for i in range(len(X) - window_size + 1):
        Xs.append(X[i:i+window_size])
        ys.append(y[i+window_size-1])  # label from last time step in window
    return np.array(Xs), np.array(ys)



# balance small classes in df_labeled by downsampling majority classes

def merge_small_classes(df, label_col='label', threshold=10, other_label='other'):
    class_counts = df[label_col].value_counts()
    small_classes = class_counts[class_counts < threshold].index
    df_merged = df.copy()
    df_merged[label_col] = df_merged[label_col].apply(lambda x: other_label if x in small_classes else x)
    return df_merged

# balance the classes in labeled_df by downsampling majority classes
def balance_classes(df, label_col='label', random_state=42):
    class_counts = df[label_col].value_counts()
    min_count = class_counts.min()

    balanced_frames = []
    for cls in class_counts.index:
        cls_df = df[df[label_col] == cls]
        balanced_cls_df = resample(cls_df, 
                                   replace=False, 
                                   n_samples=min_count, 
                                   random_state=random_state)
        balanced_frames.append(balanced_cls_df)
    
    # Concatenate and shuffle
    balanced_df = pd.concat(balanced_frames).sample(frac=1, random_state=random_state).reset_index(drop=True)
    return balanced_df

In [105]:
df_removed_nans_forecasting, df_removed_nans_classification = get_data()

Getting values...
Numeric columns: (39435, 48)
Status columns: (39435, 22)
Preprocessing data...


In [106]:
df_classification = pd.merge(
    df_removed_nans_forecasting, 
    df_removed_nans_classification,
    on='timestamp', 
    how='inner'
)

In [109]:
encoded_column_names, port_numbers = encode_port_column_names_improved(df_classification.columns[1:])
encoded_column_names = ['timestamp'] + encoded_column_names

In [110]:
df_classification.columns = encoded_column_names
df_labeled, temp_bit = encode_binary_improved(df_classification)

In [116]:
# Pre-compute columns outside the apply function

temp_bit = 8
cpu_bit = 9

status_columns = [col for col in df_classification.columns if 'interface_status' in col.lower()]
temp_candidates = [col for col in df_classification.columns if 'temperature' in col.lower()]
cpu_candidates = [col for col in df_classification.columns if 'cpu' in col.lower()]

temperature_column = temp_candidates[0] if temp_candidates else None
cpu_column = cpu_candidates[0] if cpu_candidates else None

# Pre-extract port numbers for better performance
port_mapping = {}
for col in status_columns:
    if 'interface_status_' in col:
        port_num = int(col.replace('interface_status_', ''))
        port_mapping[col] = port_num

def create_binary_encoding(row):
    down_ports = []

    for col in status_columns:
        port_num = int(col.replace('interface_status_',''))
        status_value = row[col]
        if status_value == 2:
            down_ports.append(port_num)
    
    # encode statuses 
    binary_value = 0
    for port in down_ports:

        # Initial: binary_value = 0 (00000000)

        # Port 1 down:
        # binary_value |= (1 << 0)  →  0 | 1  →  00000001 (decimal 1)

        # Port 3 down:
        # binary_value |= (1 << 2)  →  1 | 4  →  00000101 (decimal 5)

        # Port 5 down:
        # binary_value |= (1 << 4)  →  5 | 16 →  00010101 (decimal 21)
        
        binary_value |= (1 << port - 1)

    # simple temperature check (OK/NOK)
    if temperature_column and not pd.isna(row[temperature_column]):
        if row[temperature_column] > 20:
            binary_value |= (1 << temp_bit)   

    # simple CPU check (OK/NOK)
    if cpu_column and not pd.isna(row[cpu_column]):
        if row[cpu_column] > 20:
            binary_value |= (1 << cpu_bit)

    return binary_value

df_labeled = df_classification.copy()
df_labeled['binary_status'] = df_classification.apply(create_binary_encoding, axis=1)

In [117]:
# label decoder

def decode_binary(label, port_numbers, temp_bit, cpu_bit):
    result = {}
    # everything is OK
    if label == 0:
        result = {
            "down_ports": [],
            "temperature": "OK",
            "cpu": "OK"
        }
        return "All systems OK"
    
    # decode port statuses
    down_ports = []
    for port in port_numbers:
        if label & (1 << (port - 1)):
            down_ports.append(port)
        
    # decode temperature
    # checks if temp bit is set to 1

    if label & (1 << temp_bit):
        temperature_status = 'NOK'
    else:
        temperature_status = 'OK'

    # decode cpu

    if label & (1 << cpu_bit):
        cpu_status = 'NOK'
    else:
        cpu_status = 'OK'

    result = {
        "down_ports": down_ports,
        "temperature": temperature_status,
        "cpu": cpu_status
    }

    ports = []
    
    if result['down_ports']:
        if len(result['down_ports']) == 1:
            ports.append(f"Port {result['down_ports'][0]} down")
        else:
            ports_str = ', '.join(map(str, result['down_ports']))
            ports.append(f"Ports {ports_str} down")
    
    if result['temperature'] == 'NOK':
        ports.append("Temperature NOK")
    
    return "; ".join(ports)

In [118]:
label_to_name = {}

labels = df_labeled['binary_status']
for unique_label in labels.unique():
    print(unique_label)
    print(decode_binary(unique_label, port_numbers, temp_bit, cpu_bit))
    label_to_name[int(unique_label)] = decode_binary(unique_label, port_numbers, temp_bit, cpu_bit)

221757450107740658024790768743661023122968542207910901339652352
Ports 102, 104, 107, 108, 109, 202, 204, 208 down; Temperature NOK
221757450107740658024790768743661023122968542207910901339652864
Ports 102, 104, 107, 108, 109, 202, 204, 208 down; Temperature NOK
223364388151999648300332730836002185725490745201693694174954240
Ports 102, 104, 107, 108, 109, 201, 202, 204, 208 down; Temperature NOK
223364388151999648300332730836002185725490745201693694174953728
Ports 102, 104, 107, 108, 109, 201, 202, 204, 208 down; Temperature NOK
223364388151999648300332730836003453376090973431095190878159104
Ports 101, 102, 104, 107, 108, 109, 201, 202, 204, 208 down; Temperature NOK
221757450107740658024790768743666093725369455125516888152473856
Ports 102, 103, 104, 107, 108, 109, 202, 204, 208 down; Temperature NOK
228185202284776619126958617113025673533057354183042072680857856
Ports 102, 104, 107, 108, 109, 202, 203, 204, 208 down; Temperature NOK
13385793908677388995264544229202465062984855467276549